# Assignment: Decision Trees and Random Forests on Dry Bean Classification

## Objective

A food-processing system measures the shape of individual dry beans. Your task is to build a model that classifies each bean into one of seven varieties.

This is an experiment-driven assignment. You will change one modeling choice at a time, measure what changes, and explain why.

## Learning outcomes

By completing this assignment, you should be able to:

- recognize underfitting and overfitting in a Decision Tree;
- control tree complexity using depth and leaf-size parameters;
- compare Gini impurity, entropy, and log loss as split criteria;
- study how the number of trees affects a Random Forest;
- explain why random feature sampling creates diversity;
- interpret built-in Decision Tree and Random Forest feature importance;
- select a model using validation data and evaluate it once on unseen test data.

**Expected effort:** approximately 2-3 hours.


## Dataset

We will use the **UCI Dry Bean dataset**. Images of 13,611 bean grains were processed to produce 16 numerical shape measurements. The target contains seven registered bean varieties:

`SEKER`, `BARBUNYA`, `BOMBAY`, `CALI`, `DERMASON`, `HOROZ`, and `SIRA`.

Examples of input features include area, perimeter, major-axis length, eccentricity, roundness, compactness, and shape factors.

Dataset page: <https://archive.ics.uci.edu/dataset/602/dry+bean>

The loading code is provided. Your work begins with understanding the data and building the experiments.


## Assignment rules

1. Complete every `TODO` cell and remove its `NotImplementedError`.
2. Keep `RANDOM_STATE = 42` so results remain reproducible.
3. Use the **training set** to fit models.
4. Use the **validation set** to compare settings and select the final model.
5. Do not inspect test performance while tuning.
6. Evaluate the test set only once, after making the final model choice.
7. For every experiment, change only the parameter being studied.


## 1. Setup

Run the installation cell only if a package is missing.


In [ ]:
# Run this only if required.
# %pip install numpy pandas matplotlib seaborn scikit-learn ucimlrepo


In [ ]:
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)


## 2. Load the Dry Bean Dataset


In [ ]:
# UCI dataset ID 602: Dry Bean
dry_bean = fetch_ucirepo(id=602)

X = dry_bean.data.features.copy()
y = dry_bean.data.targets["Class"].copy()

# Remove exact duplicate observations before splitting. Otherwise, identical
# rows could appear in both the training and evaluation sets.
data = pd.concat([X, y.rename("Class")], axis=1)
duplicate_count = data.duplicated().sum()
data = data.drop_duplicates().reset_index(drop=True)

X = data.drop(columns="Class")
y = data["Class"]

print("Exact duplicates removed:", duplicate_count)
print("Feature shape after cleaning:", X.shape)
print("Target shape after cleaning:", y.shape)
X.head()


In [ ]:
# Basic checks supplied for you.
print("Missing feature values:", X.isna().sum().sum())
print("Duplicate observations remaining:", pd.concat([X, y], axis=1).duplicated().sum())
print("Target classes:", sorted(y.unique()))


## 3. Understand the Data

Before fitting a model, inspect the feature ranges and class distribution.

All inputs are numerical. Decision Trees and Random Forests compare feature thresholds, so scaling is not required for these models.

### TODO

- Display descriptive statistics for all features.
- Create a table containing the count and percentage of each bean class.
- Plot the class counts.


In [ ]:
# TODO: Display descriptive statistics for the numerical features.

raise NotImplementedError("Inspect feature statistics")


In [ ]:
# TODO: Create a class-distribution table with Count and Percentage columns.
# TODO: Plot the number of observations in every class.

raise NotImplementedError("Explore class distribution")


### Question 1

1. Are all classes equally represented?
2. Why will **macro F1-score** be useful in addition to accuracy for this multiclass problem?

**Your answer:**

-


## 4. Create Train, Validation, and Test Sets

Use a **60% training, 20% validation, and 20% test** split.

Apply stratification during both splitting steps so all seven classes retain similar proportions.

The validation set is used for experiments and model selection. The test set represents completely unseen data.


In [ ]:
# TODO: First separate 20% of the full dataset as the test set.
# TODO: Then split the remaining 80% into 75% training and 25% validation.
# This produces an overall 60/20/20 split.
# Use stratification and RANDOM_STATE in both steps.

raise NotImplementedError("Create stratified train-validation-test sets")


In [ ]:
# TODO: Print the shape of all six objects.
# TODO: Verify the percentage distribution of classes in train, validation, and test targets.

raise NotImplementedError("Verify the data split")


### Question 2

Why should the test set remain untouched until the final model has been selected?

**Your answer:**

-


## 5. Evaluation Helpers

These helpers are provided so that your experiments use consistent metrics.


In [ ]:
def classification_metrics(y_true, y_pred):
    # Return metrics suitable for multiclass model comparison.
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def fit_and_measure(model, X_train, y_train, X_eval, y_eval):
    # Fit a fresh model and return train/evaluation metrics plus training time.
    fitted_model = clone(model)
    start = time.perf_counter()
    fitted_model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start

    train_pred = fitted_model.predict(X_train)
    eval_pred = fitted_model.predict(X_eval)

    return fitted_model, {
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Validation Accuracy": accuracy_score(y_eval, eval_pred),
        "Validation Macro F1": f1_score(y_eval, eval_pred, average="macro", zero_division=0),
        "Fit Time (s)": fit_seconds,
    }


## 6. Baseline Decision Tree

First train an unrestricted Decision Tree. This gives the tree enough freedom to keep splitting until its stopping rules are reached.

### Before running the model

Do you expect its training accuracy to be low or high? What do you expect on validation data?

**Your prediction:**

-


In [ ]:
# TODO: Create an unrestricted DecisionTreeClassifier using RANDOM_STATE.
# TODO: Fit it on the training set with fit_and_measure().
# TODO: display the returned metrics.

raise NotImplementedError("Train the baseline Decision Tree")


In [ ]:
# TODO: Report the fitted tree's depth and number of leaves.
# TODO: Plot only the first three levels of the tree so the figure remains readable.

raise NotImplementedError("Inspect the baseline tree")


### Question 3

Compare training and validation accuracy. What evidence of overfitting or underfitting do you observe?

**Your answer:**

-


## 7. Experiment 1: Maximum Tree Depth

`max_depth` limits how many levels the tree can grow.

Test:

```python
[1, 2, 3, 5, 10, None]
```

Keep all other model settings unchanged.

### Before running the experiment

Sketch or describe how you expect training and validation accuracy to change as depth increases.


In [ ]:
depth_values = [1, 2, 3, 5, 10, None]

# TODO: Train one Decision Tree for every max_depth value.
# TODO: Store max_depth, train accuracy, validation accuracy,
# validation macro F1, actual tree depth, and number of leaves.
# TODO: Convert the collected results into depth_results DataFrame.

raise NotImplementedError("Run the maximum-depth experiment")


In [ ]:
# TODO: Plot training accuracy and validation accuracy against max_depth.
# Represent None with a readable label such as "Unlimited".

raise NotImplementedError("Visualize the maximum-depth experiment")


### Question 4

1. Which depth underfits?
2. Where does validation performance become strong?
3. At what point does additional depth mainly improve training performance?

**Your answer:**

-


## 8. Experiment 2: Minimum Samples per Leaf

`min_samples_leaf` controls the minimum number of training observations allowed in every leaf.

Test:

```python
[1, 2, 5, 10, 20, 50]
```

For this experiment, use the single `max_depth` value that performed best in Experiment 1. Keep it fixed for every run.

### Before running the experiment

How should increasing the minimum leaf size affect tree complexity?


In [ ]:
leaf_values = [1, 2, 5, 10, 20, 50]
selected_depth = None  # TODO: Replace using Experiment 1, not test results.

# TODO: Train and evaluate one tree for every min_samples_leaf value.
# TODO: Store train accuracy, validation accuracy, validation macro F1,
# actual depth, and number of leaves in leaf_results.

raise NotImplementedError("Run the minimum-leaf-size experiment")


In [ ]:
# TODO: Create a plot that compares train and validation accuracy
# across the tested minimum leaf sizes.

raise NotImplementedError("Visualize the minimum-leaf-size experiment")


### Question 5

1. How did larger leaves affect depth and the number of leaves?
2. Which setting produced the best validation result?
3. Did making leaves very large eventually cause underfitting?

**Your answer:**

-


## 9. Experiment 3: Split Criterion

A tree needs a measure of node impurity to compare candidate splits. Compare:

```python
["gini", "entropy", "log_loss"]
```

Use your selected `max_depth` and `min_samples_leaf` values for all three models.

### Before running the experiment

Do you expect these criteria to produce identical trees? Do you expect a large or small performance difference?


In [ ]:
criteria = ["gini", "entropy", "log_loss"]
selected_leaf_size = None  # TODO: Replace using Experiment 2.

# TODO: Train and evaluate one Decision Tree for each criterion.
# TODO: Store criterion, train accuracy, validation accuracy,
# validation macro F1, depth, leaves, and fit time in criterion_results.

raise NotImplementedError("Compare Decision Tree split criteria")


### Question 6

1. Did all criteria produce the same depth and number of leaves?
2. Was one criterion clearly superior, or were the differences small?
3. Which criterion will you carry forward, and why?

**Your answer:**

-


## 10. Baseline Random Forest

A Random Forest combines many trees trained on bootstrap samples. At each split, each tree considers only a subset of features. The final prediction is obtained through voting.

Train a baseline forest with:

```python
n_estimators=100
oob_score=True
```

Keep the other parameters at their default values and use `n_jobs=-1`.


In [ ]:
# TODO: Create and train the baseline Random Forest.
# TODO: Report train accuracy, validation accuracy, validation macro F1,
# OOB score, and fit time.

raise NotImplementedError("Train the baseline Random Forest")


### Question 7

How does the baseline Random Forest compare with the unrestricted Decision Tree in validation performance and overfitting gap?

**Your answer:**

-


## 11. Experiment 4: Number of Trees

Test:

```python
[1, 5, 10, 25, 50, 100, 200]
```

For every forest, enable `oob_score=True` and keep the remaining settings unchanged.

> With only one or a few trees, some observations may receive no OOB predictions. A warning or an unstable OOB score is expected and is itself an important observation.

### Before running the experiment

Will adding more trees continuously produce large improvements, or should performance eventually stabilize?


In [ ]:
tree_counts = [1, 5, 10, 25, 50, 100, 200]

# TODO: Train one Random Forest for every n_estimators value.
# TODO: Store train accuracy, validation accuracy, validation macro F1,
# OOB score, and fit time in n_trees_results.

raise NotImplementedError("Run the number-of-trees experiment")


In [ ]:
# TODO: Plot validation accuracy and OOB score against number of trees.
# TODO: Create a second plot for fit time against number of trees.

raise NotImplementedError("Visualize the number-of-trees experiment")


### Question 8

1. Around what number of trees did validation performance begin to stabilize?
2. How reliable was the OOB score for very small forests?
3. What is the cost of continuing to add trees after performance stabilizes?

**Your answer:**

-


## 12. Experiment 5: Number of Features Considered per Split

Compare:

```python
[1.0, "sqrt", "log2", 0.5]
```

Use the same selected number of trees in every run.

- `1.0`: consider all features at each split;
- `"sqrt"`: consider approximately the square root of the feature count;
- `"log2"`: consider approximately the base-2 logarithm of the feature count;
- `0.5`: consider half of the features.

### Before running the experiment

What might happen to tree diversity when every split can consider all features?


In [ ]:
feature_options = [1.0, "sqrt", "log2", 0.5]
selected_n_estimators = None  # TODO: Replace using Experiment 4.

# TODO: Train one Random Forest for every max_features option.
# TODO: Store train accuracy, validation accuracy, validation macro F1,
# OOB score, and fit time in max_features_results.

raise NotImplementedError("Run the feature-sampling experiment")


### Question 9

1. Which setting produced the strongest validation macro F1?
2. Did using all features at every split necessarily produce the best forest?
3. Explain how random feature sampling can make trees less correlated.

**Your answer:**

-


## 13. Experiment 6: Decision Tree and Random Forest Feature Importance

For a fitted tree-based model, `feature_importances_` measures the normalized total impurity reduction attributed to each feature.

This importance is model-specific. It does not establish causality, and correlated features may share or redistribute importance.

Use your selected Decision Tree and Random Forest settings. Compare their ten most important features.

Then use the Random Forest ranking to train forests with:

- the top 5 features;
- the top 10 features;
- all 16 features.

Do all comparisons on the validation set. Do not use the test set.

### Before running the experiment

Do you expect one Decision Tree and an ensemble of trees to produce identical importance rankings? Why?


In [ ]:
# TODO: Create and fit your selected Decision Tree and selected Random Forest.
# TODO: Build one importance DataFrame for each model.
# Each table should contain Feature and Importance columns, sorted descending.

raise NotImplementedError("Extract Decision Tree and Random Forest feature importance")


In [ ]:
# TODO: Plot the top 10 features from each model side by side.

raise NotImplementedError("Compare the feature-importance rankings")


In [ ]:
# TODO: Using the Random Forest ranking, evaluate forests trained with
# the top 5 features, top 10 features, and all features.
# Store feature count, selected feature names, validation accuracy,
# and validation macro F1 in reduced_feature_results.

raise NotImplementedError("Run the reduced-feature experiment")


### Question 10

1. Which features appeared near the top for both models?
2. Why can importance from one unrestricted tree be less stable than importance aggregated across a forest?
3. How much performance was retained using only the top 5 or top 10 features?
4. Does a low importance prove that a feature has no value in every possible model? Explain.

**Your answer:**

-


## 14. Experiment Summary

Create a compact summary containing the most important result from every experiment.


In [ ]:
# TODO: Complete this experiment summary.
experiment_summary = pd.DataFrame([
    {"Experiment": "Maximum depth", "Selected setting": None, "Validation Macro F1": None, "Observation": ""},
    {"Experiment": "Minimum leaf size", "Selected setting": None, "Validation Macro F1": None, "Observation": ""},
    {"Experiment": "Split criterion", "Selected setting": None, "Validation Macro F1": None, "Observation": ""},
    {"Experiment": "Number of trees", "Selected setting": None, "Validation Macro F1": None, "Observation": ""},
    {"Experiment": "Features per split", "Selected setting": None, "Validation Macro F1": None, "Observation": ""},
    {"Experiment": "Feature subset", "Selected setting": None, "Validation Macro F1": None, "Observation": ""},
])

experiment_summary


## 15. Final Model Selection and Test Evaluation

Choose either a Decision Tree or Random Forest using only the validation evidence collected above.

Write down your final choice **before** evaluating the test set.

### Final choice

- Model:
- Hyperparameters:
- Features used:
- Validation evidence supporting this choice:


In [ ]:
# TODO: Combine the training and validation data.
# TODO: Recreate the selected model with the chosen hyperparameters.
# TODO: Fit it on the combined training + validation data.
# TODO: Predict the untouched test set exactly once.

raise NotImplementedError("Train and test the final model")


In [ ]:
# TODO: Report test accuracy, macro precision, macro recall, and macro F1.
# TODO: Print the full classification report.
# TODO: Plot the confusion matrix.

raise NotImplementedError("Evaluate the final model")


## 16. Final Reflection

Answer briefly and in your own words.

### Question 11

Where did you observe underfitting? Where did you observe overfitting?

### Question 12

Why did the Random Forest generally behave differently from a single Decision Tree?

### Question 13

Which change had the largest effect on validation performance?

### Question 14

Did the final test result agree with the validation result? What would a large difference between them suggest?

### Question 15

If prediction speed or interpretability mattered more than a small accuracy improvement, could you choose a different final model? Explain.

**Your answers:**

-


## Submission Checklist

Submit the completed notebook with:

- all TODO cells completed;
- all code cells executed and outputs visible;
- train/validation/test class-distribution checks;
- baseline Decision Tree and Random Forest results;
- result tables and plots for all six experiments;
- Decision Tree and Random Forest feature-importance plots;
- the completed experiment summary;
- final test metrics, classification report, and confusion matrix;
- answers to all 15 questions;
- a short conclusion naming your final model and the evidence behind it.
